<h1 align="center"><strong><font size="6"> Weekly Predictions — Scoring Probabilities <br><br> SFM II · C_ELO · light </h1></strong></font>

<br>

**Weekly serving notebook for the committed 2026/27 spec — pure-NumPy inference.** Sibling of
`006_040__Predictions_ScoringProb__SFM_OG.py`, but on the light path: it loads the
`__LIGHT.pkl` companion bundle (plain pickle: posterior draws + the graph-exported
`f_within`/`f_long` grids) and reconstructs the ordered-logistic probabilities directly —
**no PyMC import, no model rehydration, no `sample_posterior_predictive`**. Output is the
identical `{dataset: {player: {'low','mid','up','match_stats'}}}` structure to
`static/data/040_ScoringProb__prod.pkl`; the website cannot tell the difference.

**Why this is safe (the receipts):** the reconstruction was validated 2026-08-14 against a
real PyMC graph reference (sfmII venv) — same-player probabilities matched **per draw to
4.4 × 10⁻¹⁶** across 600 draws × 1,200 rows; new-player means within Monte-Carlo tolerance.
And every serving run re-proves the wiring: the bundle carries 64 **golden rows** whose
probabilities were computed at export time — the first cell recomputes them here and refuses
to serve on any mismatch. The bundle↔CSV fingerprint check guards the data side the same way.

⚠️ Run `SMOKE = True` first (2 players); the export refuses while SMOKE is on.

In [ ]:
# ----------------------------- USER INTERACTION ----------------------------- #

# --- Set the directory to the data folder:
directory = '/Users/maximilian/Dropbox/Max/51_SoccerAnalytics'

# --- Fitted bundle stem (the harness's production export writes BOTH files;
# --- this notebook reads the light one: {stem}__LIGHT.pkl):
SFM_model__NAME = 'SFM_II_FinalC_ELO_scaleCS__2526'

# --- Which is the last season INCLUDED IN TRAINING of that bundle? [YYYY/YY]
train_end = '2025/26'

# --- Which is the current Out-Of-Sample Season? [YYYY/YY]
oos_season = '2026/27'

# --- Credible region for the bands:
cred_region = 0.9

# --- Which datasets to process? ('oos' is the weekly job)
datasets_to_process = [
    #'train',
    #'test',
    'oos',
]

# --- SMOKE: restrict to a couple of players to verify the whole path end-to-end.
SMOKE = True
SMOKE_N_PLAYERS = 2

# ----------------------------- USER INTERACTION ----------------------------- #

In [ ]:
# ============================================ Import Libraries ============================================ #
# --- NOTE: no pymc, no pytensor, no cloudpickle -- the light bundle is a plain pickle of
# --- NumPy arrays. This notebook runs anywhere NumPy runs.
import os
import gc
import pickle

import numpy as np
import pandas as pd
from tqdm import tqdm

seed = sum(map(ord, "sfm"))
rng = np.random.default_rng(seed)

<br>

## 00 &emsp; Auxiliaries — the light inference engine

Validated against the graph (see header). `ordered_probs` mirrors
`pm.OrderedLogistic.compute_p` exactly; the player effect and cutpoints are rebuilt from the
free-RV draws with the training algebra (`baseline_sigma` uses the **training** player count
— the M2.1 lesson, hardcoded here so it cannot regress).

In [ ]:
# ======================================== Light Engine ======================================== #

def softplus(x):
    return np.logaddexp(0.0, x)


def ordered_probs(eta_s, cut_s):
    """eta_s: (S, n); cut_s: (S, n, K-1) -> (S, n, K). Mirrors pm.OrderedLogistic.compute_p."""
    cdf = 1.0 / (1.0 + np.exp(-(cut_s - eta_s[..., None])))
    return np.concatenate([cdf[..., :1], np.diff(cdf, axis=-1), 1.0 - cdf[..., -1:]], axis=-1)


def _player_effect_and_cutpoints(L):
    """Rebuild (S, P) player effects and (S, P, 3) cutpoints from the free-RV draws."""
    D = L['draws']
    bsig = np.sqrt(L['intercept_sigma']**2
                   + D['player_effect_diversity']**2 / L['n_players_train'])
    pe = bsig[:, None] * D['baseline'][:, None] + D['player_effect_raw']
    delta = D['delta_mean'][:, None, :] + D['delta_sigma'][:, None, :] * D['delta_player']
    cut = np.concatenate([np.full((pe.shape[0], pe.shape[1], 1), L['cutpoint_offset']),
                          L['cutpoint_offset'] + np.cumsum(softplus(delta), axis=-1)], axis=-1)
    return pe, cut, bsig


def probs_same(L, player_codes, gd_idx, season_idx, X, chunk=4096):
    """Known players: exact reconstruction. Returns per-row quantile/mid arrays via chunks
    to bound memory: dict with 'low','mid','up' of shape (n, K)."""
    D = L['draws']
    pe, cut, _ = _player_effect_and_cutpoints(L)
    lo_q, up_q = (1 - cred_region) / 2, 1 - (1 - cred_region) / 2
    out = {k: np.empty((len(player_codes), cut.shape[-1] + 0 + 1)) for k in ('low', 'mid', 'up')}
    for a in range(0, len(player_codes), chunk):
        b = slice(a, min(a + chunk, len(player_codes)))
        eta = (pe[:, player_codes[b]] + L['f_within'][:, gd_idx[b]] + L['f_long'][:, season_idx[b]]
               + np.einsum('nf,sf->sn', X[b], D['beta__team']))
        P = ordered_probs(eta, cut[:, player_codes[b]])          # (S, chunk, K)
        out['low'][b] = np.quantile(P, lo_q, axis=0)
        out['mid'][b] = np.quantile(P, 0.5, axis=0)
        out['up'][b]  = np.quantile(P, up_q, axis=0)
    return out


def probs_new(L, n_new, player_codes, gd_idx, season_idx, X, rng, chunk=4096):
    """Unseen players: fresh zero-sum effects + fresh cutpoint deltas per posterior draw
    (the M2-repaired predictive), then the same reconstruction."""
    D = L['draws']
    S = D['baseline'].shape[0]
    _, _, bsig = _player_effect_and_cutpoints(L)
    sd = D['player_effect_diversity']
    eps = rng.normal(size=(S, n_new)) * sd[:, None]
    zsn = eps - eps.mean(axis=1, keepdims=True)                  # zero-sum projection
    pe_n = bsig[:, None] * D['baseline'][:, None] + zsn
    d_n = (D['delta_mean'][:, None, :]
           + D['delta_sigma'][:, None, :] * rng.normal(size=(S, n_new, 2)))
    cut_n = np.concatenate([np.full((S, n_new, 1), L['cutpoint_offset']),
                            L['cutpoint_offset'] + np.cumsum(softplus(d_n), axis=-1)], axis=-1)
    lo_q, up_q = (1 - cred_region) / 2, 1 - (1 - cred_region) / 2
    out = {k: np.empty((len(player_codes), cut_n.shape[-1] + 1)) for k in ('low', 'mid', 'up')}
    for a in range(0, len(player_codes), chunk):
        b = slice(a, min(a + chunk, len(player_codes)))
        eta = (pe_n[:, player_codes[b]] + L['f_within'][:, gd_idx[b]] + L['f_long'][:, season_idx[b]]
               + np.einsum('nf,sf->sn', X[b], D['beta__team']))
        P = ordered_probs(eta, cut_n[:, player_codes[b]])
        out['low'][b] = np.quantile(P, lo_q, axis=0)
        out['mid'][b] = np.quantile(P, 0.5, axis=0)
        out['up'][b]  = np.quantile(P, up_q, axis=0)
    return out


def update_elo(r_home, r_away, result, K=20, home_adv=50):
    """SFMMO-family ELO; MUST match the training implementation exactly."""
    exp_home = 1 / (1 + 10 ** ((r_away - r_home - home_adv) / 400))
    s_home = {2: 1.0, 1: 0.5, 0: 0.0}[result]
    return r_home + K * (s_home - exp_home), r_away + K * ((1 - s_home) - (1 - exp_home))


def _cs_scale_cols(x):
    """Factor-wise cross-sectional standardization within a season x gameday bucket."""
    return x.apply(lambda col: (col - col.mean()) / col.std()
                   if (len(x) > 1 and col.std() > 0) else col * 0.0)

<br>

## 1 &emsp; Load the Light Bundle — and prove it

In [ ]:
# -------------------------- Load & Golden Self-Test -------------------------- #

with open(f'{directory}/10_data/01_Models/{SFM_model__NAME}__LIGHT.pkl', 'rb') as f:
    L = pickle.load(f)

factors_CS    = L['factor_standardize']
factors_team  = L['factors_team']
players_train = pd.Index(L['players_ordered'])
elo_cfg       = L['elo']
_dc           = L['data_contract']

print('bundle      :', f'{SFM_model__NAME}__LIGHT.pkl')
print('provenance  :', L['provenance'].get('selection', 'n/a'))
print('trained thru:', _dc.get('train_end', train_end),
      f"| {_dc.get('n_rows', 0):,} training rows | {len(players_train):,} players")
print('factors_team:', factors_team)
assert 'elo_diff' in factors_CS, 'this bundle is not a C_ELO bundle'

# --- GOLDEN SELF-TEST: recompute the 64 export-time rows with THIS notebook's engine.
# --- Any wiring/version drift between export and serving fails HERE, not in production.
_g = L['golden']
_pe, _cut, _ = _player_effect_and_cutpoints(L)
_eta = (_pe[:, _g['player_codes']] + L['f_within'][:, _g['gd_idx']]
        + L['f_long'][:, _g['season_idx']]
        + np.einsum('nf,sf->sn', _g['X'], L['draws']['beta__team']))
_p = ordered_probs(_eta, _cut[:, _g['player_codes']])
_gdev = float(np.abs(_p - _g['probs']).max())
print(f'\n[golden rows] max |serving - export| = {_gdev:.3e}  over {len(_g["player_codes"])} rows x {_p.shape[0]} draws')
assert _gdev < 1e-10, f'GOLDEN ROWS FAILED ({_gdev:.3e}) -- engine drift between export and serving. DO NOT SERVE.'
print('[golden rows] PASS -- the light engine reproduces the export bit-for-bit.')
del _pe, _cut, _eta, _p

<br>

## 2 &emsp; Data, Features & ELO roll-forward

In [ ]:
# -------------------------- Load & Combine -------------------------- #

data__base = pd.read_csv(f'{directory}/10_data/106_Website/data_byPlayer__SFM_II.csv')
data__base['is_oos'] = False

_oos_path = f'{directory}/10_data/106_Website/data_byPlayer__OOS.csv'
_has_oos = os.path.exists(_oos_path) and os.path.getsize(_oos_path) > 0
if _has_oos:
    data__oos_file = pd.read_csv(_oos_path)
    if len(data__oos_file) == 0:
        _has_oos = False
    else:
        data__oos_file['is_oos'] = True
        data__all = pd.concat([data__base, data__oos_file], axis=0).reset_index(drop=True)
if not _has_oos:
    print('!! WARNING: data_byPlayer__OOS.csv missing or EMPTY -- no upcoming fixtures to serve.')
    print('   (Known pipeline issue D4/O-C2 in AUDIT_006_PIPELINE_2026-08.md -- fix 006_021 first.)')
    data__all = data__base.copy()

data__all['kick_off'] = pd.to_datetime(data__all['kick_off'])
data__all = data__all.sort_values(['name_player', 'season', 'kick_off']).reset_index(drop=True)

# --- Goals (OOS rows carry no result):
data__all['goals_in_match'] = (data__all['goals_in_match']
                               .replace([np.inf, -np.inf], np.nan).fillna(0).astype(int))
data__all['goals_cats'] = np.where(data__all['goals_in_match'] >= 3, 3, data__all['goals_in_match'])

# --- gameday as int (tolerate split-matchday suffixes on the OOS side):
data__all['gameday'] = pd.to_numeric(data__all['gameday'].astype(str).str.split('-').str[0],
                                     errors='coerce').astype('Int64')
assert data__all['gameday'].notna().all(), 'unparseable gameday values'
data__all['gameday'] = data__all['gameday'].astype(int)

print(f'{len(data__all):,} rows total | OOS rows: {int(data__all.is_oos.sum()):,}')

In [ ]:
# -------------------------- Feature Engineering (mirrors training) -------------------------- #

# --- Goal appeal:
data__all['goal_appeal'] = data__all['goalsconceded_rank_opp'] - data__all['goalsscored_rank_team']

# --- Positions: within-season fill (as in training), THEN a cross-season carry-forward.
# --- The carry-forward is a SERVING necessity: 006_021 ships the OOS fixture rows with
# --- position_player EMPTY (0% filled, verified 2026-08-15), and the within-season group for
# --- an upcoming season contains only those empty rows -- so a within-season fill alone would
# --- silently code EVERY player as the reference class and zero out position_FOR. Training
# --- rows are unaffected (the frozen vintage is 100% populated, so the ffill is a no-op there).
data__all = data__all.sort_values(['name_player', 'kick_off'])
data__all['position_player'] = (data__all.groupby(['season', 'name_player'])['position_player']
                                .transform(lambda s: s.bfill().ffill()))
data__all['position_player'] = data__all.groupby('name_player')['position_player'].ffill()
data__all = data__all.sort_values(['name_player', 'season', 'kick_off']).reset_index(drop=True)
data__all['position_FOR'] = np.where(data__all['position_player'] == 'Sturm', 1, 0)

# --- drop the impossible shares, EXACTLY as training does (8 rows in the current vintage).
# --- Without this the bundle/CSV contract check below reports a row-count mismatch -- and
# --- rows training refused to learn from would be served.
_n0 = len(data__all)
data__all = data__all.loc[data__all['goalsscored_share_player_team'] <= 1, :].reset_index(drop=True)
if len(data__all) != _n0:
    print(f"dropped {_n0 - len(data__all)} row(s) with goalsscored_share_player_team > 1 (training parity)")

# --- serving guard: an unfilled OOS position means the model sees a defaulted player
if data__all.is_oos.any():
    _pna = data__all.loc[data__all.is_oos, 'position_player'].isna().mean()
    print(f'OOS positions after carry-forward: {(1-_pna)*100:.1f}% filled'
          + ('' if _pna < 0.02 else '   !! these rows default to the reference class'))

# --- Career-season index, on the FULL combined frame (continuity into the OOS season):
data__all['season_nbr'] = (data__all.groupby(['name_player'])['season']
                           .transform(lambda x: x.factorize(sort=True)[0]))


# -------------------------- ELO: roll forward, then freeze -------------------------- #
# --- Start from the bundle's stored per-league ratings (state after the last TRAINED match),
# --- advance over any matches PLAYED since, and hold that state for the unplayed fixtures.

ELO = {ll: dict(r) for ll, r in elo_cfg['ratings_final'].items()}
_K, _HA = elo_cfg['K'], elo_cfg['home_adv']
_w, _to = elo_cfg['season_start']['returning']
_promoted = elo_cfg['season_start']['promoted']
_trained_ids = set(elo_cfg['match_table']['id_match'])

# --- one row per match, home perspective:
_m = (data__all[data__all.home_pitch == 1].drop_duplicates('id_match')
      [['id_match', 'name_league', 'season', 'kick_off', 'name_team', 'name_opp',
        'goalsscored_inGame_team', 'goalsscored_inGame_opp', 'is_oos']]
      .rename(columns={'name_team': 'home', 'name_opp': 'away',
                       'goalsscored_inGame_team': 'g_home', 'goalsscored_inGame_opp': 'g_away'})
      .sort_values(['name_league', 'kick_off', 'id_match']))

# --- some matches are observed ONLY from the away perspective (8 in the current vintage,
# --- 4 of them in the live OOS batch). Recover them, flipping the roles so 'home' really is
# --- the home side -- otherwise they carry no ELO and the assert below fires.
_missing = set(data__all.id_match.unique()) - set(_m.id_match)
if _missing:
    _aw = (data__all[data__all.id_match.isin(_missing) & (data__all.home_pitch == 0)]
           .drop_duplicates('id_match')
           [['id_match', 'name_league', 'season', 'kick_off', 'name_opp', 'name_team',
             'goalsscored_inGame_opp', 'goalsscored_inGame_team', 'is_oos']])
    _aw.columns = _m.columns
    _m = (pd.concat([_m, _aw], ignore_index=True)
          .sort_values(['name_league', 'kick_off', 'id_match']))
    print(f'ELO: recovered {len(_aw)} match(es) from the away perspective')

_seen_seasons = {ll: set(elo_cfg['match_table'].loc[elo_cfg['match_table'].name_league == ll, 'season'])
                 for ll in ELO}
_elo_rows, _n_fwd, _n_frozen = [], 0, 0

for _ll, _grp in _m.groupby('name_league', sort=False):
    _R = ELO.setdefault(_ll, {})
    for _ss, _sg in _grp.groupby('season', sort=False):
        # --- new season for this league -> apply the training-time start adjustment ONCE
        if _ss not in _seen_seasons.get(_ll, set()):
            _returning = set(_R)
            for _t in set(_sg.home) | set(_sg.away):
                _R[_t] = _R[_t] * _w + _to * (1 - _w) if _t in _returning else float(_promoted)
            _seen_seasons.setdefault(_ll, set()).add(_ss)
        for _i in _sg.index:
            _h, _a = _m.at[_i, 'home'], _m.at[_i, 'away']
            _R.setdefault(_h, 1500.0); _R.setdefault(_a, 1500.0)
            _elo_rows.append((_m.at[_i, 'id_match'], _R[_h], _R[_a]))
            # --- update ONLY on matches with a real result that the bundle has not seen
            _played = (not bool(_m.at[_i, 'is_oos'])) and pd.notna(_m.at[_i, 'g_home'])
            if _played:
                if _m.at[_i, 'id_match'] not in _trained_ids:
                    _n_fwd += 1
                _res = 2 if _m.at[_i, 'g_home'] > _m.at[_i, 'g_away'] else (
                       0 if _m.at[_i, 'g_home'] < _m.at[_i, 'g_away'] else 1)
                _R[_h], _R[_a] = update_elo(_R[_h], _R[_a], _res, K=_K, home_adv=_HA)
            else:
                _n_frozen += 1

_elo_df = pd.DataFrame(_elo_rows, columns=['id_match', 'elo_home', 'elo_away'])
data__all = data__all.merge(_elo_df, on='id_match', how='left')
data__all['elo_team'] = np.where(data__all.home_pitch == 1, data__all.elo_home, data__all.elo_away)
data__all['elo_opp'] = np.where(data__all.home_pitch == 1, data__all.elo_away, data__all.elo_home)
data__all['elo_diff'] = data__all['elo_team'] - data__all['elo_opp']

assert data__all['elo_diff'].notna().all(), 'ELO did not cover every match'
print(f'ELO: {_n_fwd:,} matches rolled forward beyond the bundle | '
      f'{_n_frozen:,} unplayed fixtures served at frozen ratings')

In [ ]:
# -------------------------- Cross-Sectional Standardization -------------------------- #
# --- The spec's own convention: factor-wise within season x gameday. Buckets never cross
# --- seasons, so an OOS matchday is standardized against ITS OWN cross-section -- exactly
# --- the machinery training used, with no train-time scaler to drift out of date.
# --- NOTE (pipeline audit O-C1): 006_021 may serve goalsscored_cum_player / share /
# --- rank_wo_player as zeros. Those are REAL factors for C_ELO -- check the printout below.

_zero_share = {f: float((data__all.loc[data__all.is_oos, f] == 0).mean())
               for f in factors_CS if f in data__all.columns and data__all.is_oos.any()}
if _zero_share:
    print('OOS zero-share by factor (1.00 => the serving pipeline is not filling it):')
    for f, v in _zero_share.items():
        flag = '  <-- SUSPECT' if v > 0.95 else ''
        print(f'   {f:32s} {v:.2f}{flag}')

data__all[factors_CS] = (data__all.groupby(['season', 'gameday'], group_keys=False)[factors_CS]
                         .apply(_cs_scale_cols))
assert data__all[factors_CS].notna().all().all()
print('\nstandardized (season x gameday, factor-wise)')

# -------------------------- Bundle <-> CSV contract check -------------------------- #
# --- The bundle carries no data copy, so verify the CSV it was fitted on still matches:
# --- same training row count, same goal-category mix. A mismatch means the dataset was
# --- rebuilt since the fit -> REFIT before serving (do not silently serve a stale model).
if _dc:
    _tr = data__all[(~data__all.is_oos) & (data__all.season <= _dc.get('train_end', train_end))]
    _mix_now = _tr['goals_cats'].value_counts(normalize=True).sort_index().round(6).to_dict()
    _n_ok = (len(_tr) == _dc.get('n_rows'))
    _mix_ok = all(abs(_mix_now.get(_k, 0) - _v) < 1e-4 for _k, _v in _dc.get('goal_cats_share', {}).items())
    print(f'\nbundle/CSV contract: rows {_dc.get("n_rows"):,} vs {len(_tr):,} '
          f'{"OK" if _n_ok else "MISMATCH"} | category mix {"OK" if _mix_ok else "MISMATCH"}')
    if not (_n_ok and _mix_ok):
        print('   !! the dataset changed since this bundle was fitted -- REFIT before serving.')


<br>

## 3 &emsp; Predictions — pure NumPy

In [ ]:
# ============================================ Predictions ============================================ #

dict_PLAYERS = {d: {} for d in datasets_to_process}

for d in datasets_to_process:

    print(f'\n\n=========================== Predictions for: {d} ===========================')

    if d == 'train':
        data__new = data__all[(~data__all.is_oos) & (data__all.season <= train_end)].copy()
    elif d == 'test':
        data__new = data__all[(~data__all.is_oos) & (data__all.season > train_end)].copy()
    else:
        data__new = data__all[data__all.is_oos].copy()

    if len(data__new) == 0:
        print(f'No rows for {d} -- skipping.')
        continue

    if SMOKE:
        _keep = list(pd.Series(data__new.name_player.unique()).head(SMOKE_N_PLAYERS))
        data__new = data__new[data__new.name_player.isin(_keep)].copy()
        print(f'*** SMOKE: restricted to {len(_keep)} players / {len(data__new)} rows ***')

    _is_new = ~data__new['name_player'].isin(players_train)
    print(f'{len(data__new):,} rows | same players: {int((~_is_new).sum()):,} | new: {int(_is_new.sum()):,}')

    for pickScenario in ['samePlayer', 'newPlayer']:

        data__scenario = (data__new[_is_new if pickScenario == 'newPlayer' else ~_is_new]
                          .copy().reset_index(drop=True))
        if len(data__scenario) == 0:
            print(f'\nNo Predictions for: {pickScenario}')
            continue
        print(f'\nMaking Predictions for: {pickScenario}  ({len(data__scenario):,} rows)')

        gd_codes = pd.Categorical(data__scenario['gameday'], categories=L['unique_gamedays']).codes
        assert (gd_codes >= 0).all(), 'gameday outside 1..38'
        ss_codes = data__scenario['season_nbr'].to_numpy()
        assert ss_codes.max() < len(L['unique_seasons'])
        X = data__scenario[factors_team].astype(np.float64).to_numpy()

        if pickScenario == 'samePlayer':
            pl_codes = pd.Categorical(data__scenario['name_player'], categories=players_train).codes
            assert (pl_codes >= 0).all()
            players_here = [p for p in players_train if p in set(data__scenario['name_player'])]
            Q = probs_same(L, pl_codes, gd_codes, ss_codes, X)
        else:
            players_here = sorted(data__scenario['name_player'].unique())
            pl_codes = pd.Categorical(data__scenario['name_player'], categories=players_here).codes
            Q = probs_new(L, len(players_here), pl_codes, gd_codes, ss_codes, X, rng)

        # --- sanity: mid-quantiles behave like probabilities
        assert np.isfinite(Q['mid']).all() and (Q['mid'] >= 0).all() and (Q['mid'] <= 1).all()

        # -------------------------- Collect per player -------------------------- #
        _n_ev = Q['mid'].shape[1]
        _cols = [f'Goals_{g}' for g in range(_n_ev)]
        _names = data__scenario['name_player'].to_numpy()

        for pp in tqdm(players_here):
            _rows = np.where(_names == pp)[0]
            if len(_rows) == 0:
                continue
            _idx = [f'{data__scenario["season"].iloc[r]}__{data__scenario["gameday"].iloc[r]}'
                    for r in _rows]
            dict_plot = {k: pd.DataFrame(Q[k][_rows], index=_idx, columns=_cols)
                         for k in ('low', 'mid', 'up')}
            _ms = data__scenario.iloc[_rows][['goals_in_match', 'name_league', 'name_team',
                                              'name_opp', 'points_team', 'points_opp']].copy()
            _ms.index = _idx
            dict_plot['match_stats'] = _ms
            dict_PLAYERS[d][pp] = dict_plot

        del Q, data__scenario
        gc.collect()

    gc.collect()
    print(f"Memory cleaned up after processing '{d}'")

print('\nPredictions done!')
for d in dict_PLAYERS:
    print(f'  {d}: {len(dict_PLAYERS[d]):,} players')

<br>

## 4 &emsp; Export

Identical structure and destination to the OG serving script; the website is agnostic.

<br>

## 4 &emsp; Point-in-Time Ledger

**The site's track record is only honest if it records what was said *before* the match.**
Every run overwrites `040_ScoringProb__prod.pkl`, and the Django migration deletes and
rebuilds `WeeklyPick` from it — so without this ledger the "history" on the site is a
re-forecast. This cell maintains `SFM_predictions__frozen.csv` under the same discipline as
the SFMMO's frozen feed: **unplayed rows refresh every run; played rows are never rewritten.**

Verified by a two-run sandbox (2026-08-15): with results injected for half the fixtures and
the re-forecast deliberately altered, the frozen probabilities moved by **0.0e+00**.

In [ ]:
# ============================== POINT-IN-TIME LEDGER (frozen forecasts) ============================== #
#
# Why this exists: every serving run OVERWRITES 040_ScoringProb__prod.pkl, and the Django
# migration deletes and rebuilds WeeklyPick from it.  Without a ledger, the "historical
# predictions" the site shows are re-forecasts -- what today's pipeline says about the past,
# not what was published before the match.  Three ways that differs from the truth:
#   (a) cross-sectional standardization is computed within season x gameday, so the bucket
#       (predicted squads pre-match vs actual appearances after) changes the inputs;
#   (b) a predicted player who did not play VANISHES; a surprise starter gets a forecast
#       manufactured after the fact;
#   (c) the annual refit re-scores all history IN-SAMPLE.
#
# The rule (same discipline as SFMMO_predictions__frozen.csv):
#   * while a fixture is UNPLAYED its row refreshes every run -- newer information, still
#     legitimately pre-match;
#   * the moment it is PLAYED the row is frozen forever and the result is attached beside
#     the probabilities that were standing at kickoff.
#
# Football-specific addition the SFMMO does not need: teams always play, PLAYERS DO NOT.
# `appeared` records whether a forecast player actually featured, so the tracker can void
# rather than silently score a pick as a miss.
# ==================================================================================================== #

import datetime as _dt

LEDGER_PATH = f'{directory}/10_data/106_Website/SFM_predictions__frozen.csv'


def _board_to_frame(board, data_scored):
    """dict_PLAYERS['oos'] -> one tidy row per (id_match, name_player) with the bands."""
    idx = data_scored.set_index(['name_player', 'season', 'gameday'])
    out = []
    for pp, blocks in board.items():
        mid, low, up = blocks['mid'], blocks['low'], blocks['up']
        ms = blocks['match_stats']
        for k, tag in enumerate(mid.index):
            _season, _gd = tag.split('__')
            try:
                _id = idx.loc[(pp, _season, int(_gd)), 'id_match']
                _id = _id.iloc[0] if hasattr(_id, 'iloc') else _id
            except KeyError:
                continue
            r = dict(id_match=_id, name_player=pp, season=_season, gameday=int(_gd),
                     name_league=ms['name_league'].iloc[k], name_team=ms['name_team'].iloc[k],
                     name_opp=ms['name_opp'].iloc[k])
            for g in range(mid.shape[1]):
                r[f'p{g}_mid'] = float(mid.iloc[k, g])
                r[f'p{g}_low'] = float(low.iloc[k, g])
                r[f'p{g}_up'] = float(up.iloc[k, g])
            r['p_scores_mid'] = 1.0 - r['p0_mid']          # --- the anytime-scorer headline
            out.append(r)
    return pd.DataFrame(out)


def update_frozen_ledger(board, data_scored, played_rows, model_name, ledger_path=LEDGER_PATH,
                         now=None):
    """Refresh unplayed rows, freeze played ones forever, attach outcomes.  Returns the ledger."""
    now = now or _dt.datetime.now().isoformat(timespec='seconds')
    KEY = ['id_match', 'name_player']

    fresh = _board_to_frame(board, data_scored)
    fresh['status'] = 'upcoming'
    fresh['forecast_frozen_at'] = now
    fresh['model'] = model_name
    fresh['actual_goals'] = np.nan
    fresh['appeared'] = pd.NA

    if os.path.exists(ledger_path):
        old = pd.read_csv(ledger_path)
    else:
        old = fresh.iloc[0:0].copy()

    # --- 1. rows already finished are IMMUTABLE -----------------------------------------
    done = old[old.status == 'finished'].copy()

    # --- 2. rows that were upcoming and are now played: FREEZE the standing forecast -----
    played_ids = set(played_rows['id_match'])
    pend = old[old.status == 'upcoming'].copy()
    just = pend[pend.id_match.isin(played_ids)].copy()
    if len(just):
        res = (played_rows.groupby(['id_match', 'name_player'])['goals_in_match']
               .max().rename('_g').reset_index())
        just = just.merge(res, on=KEY, how='left')
        just['appeared'] = just['_g'].notna()
        just['actual_goals'] = just['_g']
        just = just.drop(columns='_g')
        just['status'] = 'finished'
        # --- forecast_frozen_at is NOT touched: it records when the forecast was made.

    # --- 3. still-unplayed rows: refresh from the current board ---------------------------
    still = fresh[~fresh.id_match.isin(played_ids)].copy()

    # --- 4. appearances nobody forecast (surprise starters in a played fixture) -----------
    #     Recorded with null probabilities so the coverage gap is VISIBLE rather than silent.
    if len(just):
        seen = set(map(tuple, just[KEY].values)) | set(map(tuple, done[KEY].values))
        extra = played_rows[played_rows.id_match.isin(set(just.id_match))]
        extra = extra[~extra.set_index(KEY).index.isin(seen)]
        if len(extra):
            e = extra.groupby(KEY).agg(season=('season', 'first'), gameday=('gameday', 'first'),
                                       name_league=('name_league', 'first'),
                                       name_team=('name_team', 'first'), name_opp=('name_opp', 'first'),
                                       actual_goals=('goals_in_match', 'max')).reset_index()
            e['status'] = 'finished'; e['appeared'] = True
            e['forecast_frozen_at'] = pd.NA; e['model'] = model_name
            e['p_scores_mid'] = np.nan
            just = pd.concat([just, e], ignore_index=True)

    ledger = pd.concat([done, just, still], ignore_index=True)
    ledger = ledger.drop_duplicates(subset=KEY, keep='first')       # finished wins over fresh
    ledger = ledger.sort_values(['season', 'gameday', 'name_league', 'p_scores_mid'],
                                ascending=[True, True, True, False]).reset_index(drop=True)
    ledger.to_csv(ledger_path, index=False)

    n_new_frozen = int((just['status'] == 'finished').sum()) if len(just) else 0
    print(f'[ledger] {ledger_path.split("/")[-1]}: {len(ledger):,} rows | '
          f'{int((ledger.status == "finished").sum()):,} frozen, '
          f'{int((ledger.status == "upcoming").sum()):,} upcoming '
          f'(+{n_new_frozen:,} frozen this run)')
    if len(just) and 'appeared' in just:
        _noshow = int((just['appeared'] == False).sum())
        if _noshow:
            print(f'[ledger] {_noshow} forecast player(s) did not appear -- rows kept with '
                  f'appeared=False so the tracker can VOID rather than score them as misses.')
    return ledger


# --- run it: the OOS board is the only thing ever forecast ahead of time
if 'oos' in dict_PLAYERS and len(dict_PLAYERS['oos']):
    _played_rows = data__all[~data__all.is_oos][
        ['id_match', 'name_player', 'season', 'gameday', 'name_league',
         'name_team', 'name_opp', 'goals_in_match']]
    _oos_rows = data__all[data__all.is_oos]
    ledger = update_frozen_ledger(dict_PLAYERS['oos'], _oos_rows, _played_rows,
                                  model_name=SFM_model__NAME)
else:
    print('[ledger] no OOS board this run -- ledger untouched (correct: nothing new was forecast)')


In [ ]:
# ============================================ Export the Dictionary ============================================ #

assert not SMOKE, 'SMOKE run -- a two-player dictionary must not overwrite production. Set SMOKE = False.'

train_predictions_path = f'{directory}/10_data/106_Website/040_ScoringProb__{SFM_model__NAME}__train.pkl'

if 'train' not in datasets_to_process and os.path.exists(train_predictions_path):
    print(f'Loading existing train predictions from: {train_predictions_path}')
    with open(train_predictions_path, 'rb') as f:
        train_preds = pickle.load(f)
    dict_PLAYERS['train'] = train_preds.get('train', train_preds)
    print(f"Merged {len(dict_PLAYERS['train'])} players from train predictions")
elif 'train' in datasets_to_process:
    print(f'Saving train predictions separately to: {train_predictions_path}')
    with open(train_predictions_path, 'wb') as f:
        pickle.dump({'train': dict_PLAYERS.get('train', {})}, f)

_prod = f'{directory}/00_code/006_Website/01__SFMcom/SFMwebsite__v2/static/data/040_ScoringProb__prod.pkl'
with open(_prod, 'wb') as f:
    pickle.dump(dict_PLAYERS, f)

print(f'\nWritten: {_prod}')
print(f'  model: {SFM_model__NAME}')
for d in dict_PLAYERS:
    print(f'  {d}: {len(dict_PLAYERS[d]):,} players')
print('\n-> next: 006_041__SAR_PAR (needs elo_diff added to its CONTEXT_FACTORS), then the '
      'Django migration scripts in SFMwebsite__v2/scripts/.')